# Phase 5 — Comparison metric (`paper/comparison_metric.md`)

Computes the three-layer comparison metric between the rule-based arm
(phase 3, `results/rule_based_query_spec.json` + `results/rule_based_ranking.csv`)
and the LLM arm (phase 4, `results/llm_runs/run_*.json`, the real,
independently-verified N=20 batch — see `claude/phase-4-llm-arm.md`
Round 10), and writes `results/metrics.csv`. No network, no LLM key
needed (see `paper/PLAN.md`'s phase table) — everything here is computed
from data already committed to the repo, so this notebook is
reproducible standalone.

## Setup

In [1]:
import json
import csv
import re
import statistics
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

RESULTS = Path("../results")
assert RESULTS.exists(), f"expected {RESULTS.resolve()} to exist"

## Parsing helper

Both arms save `operation_sequence` as a list of `str(OperationSpec)`
repr strings (e.g. `"OperationSpec(op='crs_transform', inputs={...},
params={...}, output='...')"`), not structured JSON — parse it back
into a plain dict rather than regex-scraping individual fields.

In [2]:
def parse_op(s: str) -> dict:
    m = re.match(r"OperationSpec\((.*)\)$", s)
    if not m:
        raise ValueError(f"not an OperationSpec repr: {s!r}")
    ns: dict = {}
    exec(f"d = dict({m.group(1)})", {}, ns)
    return ns["d"]

def op_name_sequence(operation_sequence: list[str]) -> tuple[str, ...]:
    return tuple(parse_op(s)["op"] for s in operation_sequence)

## Load phase 3 + phase 4 outputs

In [3]:
llm_run_files = sorted((RESULTS / "llm_runs").glob("run_*.json"))
llm_runs = [json.loads(f.read_text()) for f in llm_run_files]

with open(RESULTS / "rule_based_query_spec.json") as fh:
    rb_spec = json.load(fh)

rb_ranking_df = pd.read_csv(RESULTS / "rule_based_ranking.csv")

print(f"Loaded {len(llm_runs)} LLM runs and the rule-based reference ({len(rb_ranking_df)} districts)")
assert len(llm_runs) == 20
assert len(rb_ranking_df) == 23

Loaded 20 LLM runs and the rule-based reference (23 districts)


### Sanity check: phase 4's batch is actually clean, not just present

In [4]:
for r in llm_runs:
    assert r["generation_success"] and r["execution_success"] and not r["degenerate_ranking"] and r["error"] is None, \
        f"run {r.get('run_index')} was not clean - phase 4 is not actually done"
print("All 20 LLM runs confirmed clean: generation_success, execution_success, "
      "not degenerate_ranking, error is None.")

All 20 LLM runs confirmed clean: generation_success, execution_success, not degenerate_ranking, error is None.


## Layer 1 — Plan Agreement Rate (PAR)

In [5]:
llm_seqs = [op_name_sequence(r["operation_sequence"]) for r in llm_runs]
seq_counts = {}
for seq in llm_seqs:
    seq_counts[seq] = seq_counts.get(seq, 0) + 1
llm_reference_seq, mode_count = max(seq_counts.items(), key=lambda kv: kv[1])
llm_par = mode_count / len(llm_seqs)

rb_seq = op_name_sequence(rb_spec["operation_sequence"])
rb_par = 1.0  # by construction (comparison_metric.md) - asserted in notebook 02, n_runs_checked there

print(f"Rule-based arm PAR = {rb_par}  (reference: {len(rb_seq)} ops, asserted over "
      f"{rb_spec['n_runs_checked']} rebuild-and-rerun cycles in phase 3)")
print(f"  {rb_seq}")
print()
print(f"LLM arm PAR = {llm_par}  ({mode_count}/{len(llm_seqs)} runs match the mode sequence)")
print(f"  {llm_reference_seq}")
print()
print(f"Distinct op-name sequences seen across the 20 LLM runs: {len(seq_counts)}")

Rule-based arm PAR = 1.0  (reference: 10 ops, asserted over 5 rebuild-and-rerun cycles in phase 3)
  ('crs_transform', 'crs_transform', 'nearest_neighbor', 'crs_transform', 'nearest_neighbor', 'crs_transform', 'nearest_neighbor', 'score_features', 'rank_features', 'build_report')

LLM arm PAR = 1.0  (20/20 runs match the mode sequence)
  ('crs_transform', 'crs_transform', 'crs_transform', 'crs_transform', 'spatial_nearest', 'spatial_nearest', 'spatial_nearest', 'score_features', 'rank_features')

Distinct op-name sequences seen across the 20 LLM runs: 1


## Layer 2 — Parametric agreement

The LLM was never given weight/`max_distance` numbers (see phase 4's
design) — this checks how much it varies, and how it varies,
when it has to invent them itself.

In [6]:
def extract_score_features_factors(operation_sequence):
    ops = [parse_op(s) for s in operation_sequence]
    score_op = next(o for o in ops if o["op"] == "score_features")
    params = score_op["params"]
    return params["scoring_spec"]["factors"] if "scoring_spec" in params else params["factors"]

by_field = {}
for run in llm_runs:
    for f in extract_score_features_factors(run["operation_sequence"]):
        field = f["field"]
        by_field.setdefault(field, {"weight": [], "max_distance": []})
        by_field[field]["weight"].append(float(f["weight"]))
        by_field[field]["max_distance"].append(float(f.get("max_distance", f.get("max_distance_m"))))

layer2_rows = []
for field, vals in by_field.items():
    layer2_rows.append({
        "field": field,
        "weight_mean": statistics.mean(vals["weight"]),
        "weight_sd": statistics.pstdev(vals["weight"]) if len(vals["weight"]) > 1 else 0.0,
        "max_distance_mean": statistics.mean(vals["max_distance"]),
        "max_distance_sd": statistics.pstdev(vals["max_distance"]) if len(vals["max_distance"]) > 1 else 0.0,
        "n": len(vals["weight"]),
    })
layer2_df = pd.DataFrame(layer2_rows)
print("LLM arm - per-field weight / max_distance across all 20 runs:")
print(layer2_df.to_string(index=False))
print()
print("Rule-based arm's own fixed (given, not sampled) values, for contrast:")
for a in rb_spec["amenities"]:
    print(f"  {a['ref']:<8} weight={a['weight']}  max_distance_m={a['max_distance_m']}")

LLM arm - per-field weight / max_distance across all 20 runs:
              field  weight_mean  weight_sd  max_distance_mean  max_distance_sd  n
  distance_to_metro          1.0        0.0              800.0              0.0 20
distance_to_schools          1.0        0.0              800.0              0.0 20
  distance_to_parks          1.0        0.0              800.0              0.0 20

Rule-based arm's own fixed (given, not sampled) values, for contrast:
  metro    weight=3.0  max_distance_m=800.0
  schools  weight=2.0  max_distance_m=1000.0
  parks    weight=1.0  max_distance_m=1500.0


**Finding:** every one of 20 runs invents the *same* uniform choice —
`weight=1`, `max_distance=800` for all three amenities alike (`sd=0.0`
in every column — a real finding at N=20, not a small-sample artifact).
The rule-based arm's config is a considered modeling decision (metro
weighted highest with the shortest cutoff, parks weighted lowest with
the longest cutoff — see `paper/PLAN.md`'s "Amenities" section) that the
LLM has no way to reconstruct from the raw query alone, and does not
attempt to — it defaults to "flat" rather than guessing at domain
priorities. Worth stating plainly in the paper: the LLM arm is perfectly
*reliable* (PAR = 1.0, weight sd = 0.0) but not equivalent in
domain-informed *judgment* to the rule-based arm's deliberately unequal
weighting.

## Layer 3 — Rank Stability

In [7]:
site_names = sorted(rb_ranking_df["name"].tolist())
assert len(site_names) == 23

def score_vector(name_to_score: dict) -> np.ndarray:
    missing = [n for n in site_names if n not in name_to_score]
    if missing:
        raise ValueError(f"missing districts: {missing}")
    return np.array([name_to_score[n] for n in site_names], dtype=float)

llm_vectors = [
    score_vector({d["name"]: d["score"] for d in run["ranking"]})
    for run in llm_runs
]
rb_vector = score_vector(dict(zip(rb_ranking_df["name"], rb_ranking_df["accessibility_score"])))

pair_rhos = []
for i in range(len(llm_vectors)):
    for j in range(i + 1, len(llm_vectors)):
        rho, _p = spearmanr(llm_vectors[i], llm_vectors[j])
        pair_rhos.append(rho)

vs_rb_rhos = [spearmanr(v, rb_vector)[0] for v in llm_vectors]

rb_rank_stability = 1.0  # by construction, per comparison_metric.md
llm_rank_stability_mean = statistics.mean(pair_rhos)
llm_rank_stability_sd = statistics.pstdev(pair_rhos) if len(pair_rhos) > 1 else 0.0
vs_rb_mean = statistics.mean(vs_rb_rhos)
vs_rb_sd = statistics.pstdev(vs_rb_rhos) if len(vs_rb_rhos) > 1 else 0.0

print(f"Rule-based arm Rank Stability = {rb_rank_stability}  (by construction)")
print(f"LLM arm pairwise Rank Stability: rho mean={llm_rank_stability_mean:.6f}, "
      f"sd={llm_rank_stability_sd:.6f}  over {len(pair_rhos)} pairs")
print(f"LLM arm vs. rule-based reference: rho mean={vs_rb_mean:.6f}, sd={vs_rb_sd:.6f}")

Rule-based arm Rank Stability = 1.0  (by construction)
LLM arm pairwise Rank Stability: rho mean=1.000000, sd=0.000000  over 190 pairs
LLM arm vs. rule-based reference: rho mean=0.998966, sd=0.000000


In [8]:
# Why vs.-rule-based rho is 0.998966, not exactly 1.0, even though every
# LLM run's ranking is byte-identical to every other one: results/rule_based_ranking.csv
# stores Waehring and Hernals BOTH at accessibility_score=99.2 (rounded at
# write time in notebook 02) even though their true distance_to_metro_m
# differ (13 vs 14) - so scipy.stats.spearmanr treats them as EXACTLY TIED
# in the rule-based vector (average rank 22.5 each), while the LLM arm's
# own (higher-precision) scores rank them 22nd/23rd distinctly. That
# single tied pair is the entire gap from 1.0.
print(f"rb tie check: Waehring/Hernals rule-based scores = "
      f"{rb_ranking_df.set_index('name').loc[['Währing','Hernals'],'accessibility_score'].tolist()}")
llm_sample = llm_runs[0]
llm_scores = {d['name']: d['score'] for d in llm_sample['ranking']}
print(f"llm tie check: Waehring/Hernals LLM run_00 scores    = "
      f"{[llm_scores['Währing'], llm_scores['Hernals']]}")

rb tie check: Waehring/Hernals rule-based scores = [99.2, 99.2]
llm tie check: Waehring/Hernals LLM run_00 scores    = [99.472262, 99.436835]


**Not a real disagreement.** Both arms agree Währing and Hernals are the
two least-accessible districts, and agree on the underlying metro
distances (13m vs. 14m, same geometry). The 0.998966 vs. 1.0 gap is a
side effect of `rule_based_ranking.csv` rounding its score column to one
decimal, which happens to tie two districts the LLM arm's higher-precision
score keeps separate — a footnote for the paper, not a limitation of the
LLM arm's reliability.

## Reliability

In [9]:
n = len(llm_runs)
llm_success_rate = sum(1 for r in llm_runs if r["generation_success"] and r["execution_success"]) / n
llm_nondegenerate_rate = sum(1 for r in llm_runs if not r["degenerate_ranking"]) / n
latencies = sorted(r["latency_s"] for r in llm_runs)
llm_median_latency = statistics.median(latencies)
llm_iqr_low = latencies[n // 4]
llm_iqr_high = latencies[(3 * n) // 4]

print(f"LLM arm: success_rate={llm_success_rate:.4f}, non_degenerate_rate={llm_nondegenerate_rate:.4f}")
print(f"LLM arm: median latency={llm_median_latency:.3f}s, IQR=[{llm_iqr_low:.3f}, {llm_iqr_high:.3f}]s")
print("Rule-based arm: success_rate=1.0 (deterministic, no failure mode to measure); "
      "latency not tracked (sub-second, not the phenomenon this arm exists to characterize)")

LLM arm: success_rate=1.0000, non_degenerate_rate=1.0000
LLM arm: median latency=20.010s, IQR=[18.779, 21.698]s
Rule-based arm: success_rate=1.0 (deterministic, no failure mode to measure); latency not tracked (sub-second, not the phenomenon this arm exists to characterize)


## Final reporting table (`paper/comparison_metric.md`'s format) — save `results/metrics.csv`

In [10]:
metrics_rows = [
    {
        "arm": "rule_based",
        "n_runs": rb_spec["n_runs_checked"],
        "par": rb_par,
        "weight_sd_metro": 0.0, "weight_sd_schools": 0.0, "weight_sd_parks": 0.0,
        "max_distance_sd_metro": 0.0, "max_distance_sd_schools": 0.0, "max_distance_sd_parks": 0.0,
        "rank_stability_mean": rb_rank_stability, "rank_stability_sd": 0.0,
        "rho_vs_rule_based_mean": 1.0, "rho_vs_rule_based_sd": 0.0,
        "success_rate": 1.0, "non_degenerate_rate": 1.0,
        "median_latency_s": None, "latency_iqr_low_s": None, "latency_iqr_high_s": None,
    },
    {
        "arm": "llm",
        "n_runs": n,
        "par": llm_par,
        "weight_sd_metro": statistics.pstdev(by_field["distance_to_metro"]["weight"]) if len(by_field["distance_to_metro"]["weight"]) > 1 else 0.0,
        "weight_sd_schools": statistics.pstdev(by_field["distance_to_schools"]["weight"]) if len(by_field["distance_to_schools"]["weight"]) > 1 else 0.0,
        "weight_sd_parks": statistics.pstdev(by_field["distance_to_parks"]["weight"]) if len(by_field["distance_to_parks"]["weight"]) > 1 else 0.0,
        "max_distance_sd_metro": statistics.pstdev(by_field["distance_to_metro"]["max_distance"]) if len(by_field["distance_to_metro"]["max_distance"]) > 1 else 0.0,
        "max_distance_sd_schools": statistics.pstdev(by_field["distance_to_schools"]["max_distance"]) if len(by_field["distance_to_schools"]["max_distance"]) > 1 else 0.0,
        "max_distance_sd_parks": statistics.pstdev(by_field["distance_to_parks"]["max_distance"]) if len(by_field["distance_to_parks"]["max_distance"]) > 1 else 0.0,
        "rank_stability_mean": llm_rank_stability_mean, "rank_stability_sd": llm_rank_stability_sd,
        "rho_vs_rule_based_mean": vs_rb_mean, "rho_vs_rule_based_sd": vs_rb_sd,
        "success_rate": llm_success_rate, "non_degenerate_rate": llm_nondegenerate_rate,
        "median_latency_s": llm_median_latency, "latency_iqr_low_s": llm_iqr_low, "latency_iqr_high_s": llm_iqr_high,
    },
]
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(RESULTS / "metrics.csv", index=False)
print(f"wrote {(RESULTS / 'metrics.csv').resolve()}")
metrics_df

wrote /home/araz/Projects/Career/smart-spatial-vienna-accessibility/results/metrics.csv


,arm,n_runs,par,weight_sd_metro,weight_sd_schools,weight_sd_parks,max_distance_sd_metro,max_distance_sd_schools,max_distance_sd_parks,rank_stability_mean,rank_stability_sd,rho_vs_rule_based_mean,rho_vs_rule_based_sd,success_rate,non_degenerate_rate,median_latency_s,latency_iqr_low_s,latency_iqr_high_s
0,rule_based,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.000000,0.0,1.0,1.0,NaN,NaN,NaN
1,llm,20,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.998966,0.0,1.0,1.0,20.010023,18.779345,21.69795


## Final checks

In [11]:
check = pd.read_csv(RESULTS / "metrics.csv")
assert len(check) == 2
assert set(check["arm"]) == {"rule_based", "llm"}
rb_row = check[check["arm"] == "rule_based"].iloc[0]
llm_row = check[check["arm"] == "llm"].iloc[0]
assert rb_row["par"] == 1.0 and rb_row["rank_stability_mean"] == 1.0
assert llm_row["par"] == 1.0
assert llm_row["success_rate"] == 1.0 and llm_row["non_degenerate_rate"] == 1.0
assert 0.99 < llm_row["rho_vs_rule_based_mean"] <= 1.0
print("all checks passed")
print()
print(check.to_string(index=False))

all checks passed

       arm  n_runs  par  weight_sd_metro  weight_sd_schools  weight_sd_parks  max_distance_sd_metro  max_distance_sd_schools  max_distance_sd_parks  rank_stability_mean  rank_stability_sd  rho_vs_rule_based_mean  rho_vs_rule_based_sd  success_rate  non_degenerate_rate  median_latency_s  latency_iqr_low_s  latency_iqr_high_s
rule_based       5  1.0              0.0                0.0              0.0                    0.0                      0.0                    0.0                  1.0                0.0                1.000000                   0.0           1.0                  1.0               NaN                NaN                 NaN
       llm      20  1.0              0.0                0.0              0.0                    0.0                      0.0                    0.0                  1.0                0.0                0.998966                   0.0           1.0                  1.0         20.010023          18.779345            21.69795


## Findings for the paper

- **Reliability, once every upstream precondition is enforced, is
  essentially perfect for this setup.** PAR = 1.0 and Rank Stability =
  1.0 for the LLM arm at N=20 — every run built the identical plan shape
  and produced the identical final ranking. This is the resolution of
  phase 4's headline finding (six rounds of 20/20 degenerate batches
  before this one): the earlier failures were about missing/weak
  validation, not about the model's capability ceiling.
- **Reliability is not the same as matching the rule-based arm's
  judgment.** The LLM consistently invents a flat weighting scheme
  (equal weight, equal cutoff for metro/schools/parks) rather than the
  rule-based arm's deliberately unequal one (metro weighted highest with
  the shortest cutoff). Both are internally defensible, but they are not
  the same modeling choice, and the raw query never specified numbers —
  so this is squarely a case of the LLM filling an under-specified gap
  with a plausible but different default, not an error.
- **The two arms' final rankings still agree almost perfectly** (ρ =
  0.998966 vs. the rule-based reference), because Vienna's 21/23-way tie
  at the top dominates both rankings regardless of the exact weighting
  scheme — the weighting choice mostly only matters for the bottom two
  districts, which is exactly where the tiny rho gap comes from (a
  rounding-induced tie in the stored rule-based CSV, not a real
  disagreement).

See `paper/PLAN.md`'s "Findings to fold into the paper's discussion"
section, updated alongside this notebook.

## Next: phase 6

`notebooks/05_results.ipynb` — figures/tables for the paper, built from
`results/metrics.csv` (this notebook), `results/rule_based_ranking.csv`,
and `results/llm_runs/manifest.csv`. No network, no LLM key needed.